# Composing Subagents

Real agent workflows tend to split into smaller jobs, with a subagent taking on each one. In NOOA, a subagent is just another `Agent` instance: the parent creates it, passes explicit inputs, awaits one of its methods, and combines the result with normal Python. There is no special "workflow" config, no DAG to declare — the orchestration is whatever Python code you write.

Because of that, the "patterns" below aren't framework features; they're just ways of composing `await` calls. We'll walk through three of them:

- **Sequential composition** — call one subagent, then another, in plain Python. The order is fixed by the code you write.
- **Parallel composition** — kick off several independent subagents at once with ordinary `asyncio.gather`.
- **LLM-driven composition** — let an agent decide at runtime whether to spawn a subagent, by generating the Python that instantiates it.

Along the way, we'll build three variants of a `WeekendPlannerAgent`, one per pattern.

## Prerequisites

Run the install cell below before the setup cell.

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


In [ ]:
!pip install nooa


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below. Replace `"your-api-key"` with a real key for hosted providers; local providers such as Ollama and vLLM do not need a key, just an `api_base`.


In [ ]:
from nooa.unifiedllm.registry import get_llm_client

# model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                         # Anthropic
model = get_llm_client("gpt-5.5", api_key="your-api-key")                                    # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")        # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1") # vLLM (local, no key)


## The setting: a Weekend Planner Agent

We are going to build a weekend planner agent, that will help us plan a relaxed Saturday in Lisbon. 

We'll build two subagents:

- **`TripScoutAgent`** pulls trip preferences out of a messy traveler request.
- **`ItineraryBuilderAgent`** inspects a small city guide through helper methods and writes the plan.

The domain is toy on purpose. Swap the guide for a database and the request for a support ticket, and the shape stays the same: a parent orchestrator handing focused inputs to specialized children.

The pattern worth watching is the composition itself: the parent is a plain Python orchestrator, and each child owns one focused LLM task.

In [37]:
from nooa import Agent, strategy
from nooa.agentdoc import doc
from nooa.strategies import PredictStrategy


In [38]:
TRAVELER_REQUEST = """
I'm spending one Saturday in Lisbon with my partner. We like great food, tilework,
bookshops, river views, and a little history, but we don't want a packed schedule.
Please avoid clubs and late nights. We are happy to walk, but not all day.
""".strip()

print(TRAVELER_REQUEST)


I'm spending one Saturday in Lisbon with my partner. We like great food, tilework,
bookshops, river views, and a little history, but we don't want a packed schedule.
Please avoid clubs and late nights. We are happy to walk, but not all day.

For the itinerary builder to have something to work with, we'll give it a tiny hand-written city guide. In a real app this could be a database, an API result, or a dataframe. Here it's just enough live Python state for CodeAct to inspect through methods.

In [39]:
LISBON_GUIDE = [
    {"name": "Manteigaria", "kind": "food", "area": "Chiado", "duration": 30, "tags": ["pastry", "classic", "quick"]},
    {"name": "Museu Nacional do Azulejo", "kind": "museum", "area": "Xabregas", "duration": 90, "tags": ["tilework", "history", "quiet"]},
    {"name": "Ler Devagar", "kind": "bookshop", "area": "LX Factory", "duration": 45, "tags": ["books", "design", "coffee"]},
    {"name": "Miradouro de Santa Catarina", "kind": "viewpoint", "area": "Bica", "duration": 35, "tags": ["river views", "sunset", "relaxed"]},
    {"name": "Time Out Market", "kind": "food", "area": "Cais do Sodre", "duration": 60, "tags": ["food hall", "easy", "busy"]},
    {"name": "MAAT river walk", "kind": "walk", "area": "Belem", "duration": 60, "tags": ["river views", "architecture", "gentle walk"]},
    {"name": "Alfama wander", "kind": "walk", "area": "Alfama", "duration": 75, "tags": ["history", "hills", "classic"]},
]

for place in LISBON_GUIDE:
    print(f"- {place['name']} ({place['kind']}, {place['area']}): {', '.join(place['tags'])}")


- Manteigaria (food, Chiado): pastry, classic, quick
- Museu Nacional do Azulejo (museum, Xabregas): tilework, history, quiet
- Ler Devagar (bookshop, LX Factory): books, design, coffee
- Miradouro de Santa Catarina (viewpoint, Bica): river views, sunset, relaxed
- Time Out Market (food, Cais do Sodre): food hall, easy, busy
- MAAT river walk (walk, Belem): river views, architecture, gentle walk
- Alfama wander (walk, Alfama): history, hills, classic

## 1. A static, sequential workflow

We'll start with the simplest pattern: the parent's Python code fixes exactly which subagents run, and in what order. There is no framework construct behind this — it's just two `await` calls in a method body. Let's build the two subagents first, then wire them together in a parent orchestrator.

### Subagent 1

Our first agent, which we'll call **Trip Scout Agent**, has a narrow job: **read the traveler request and return a short list of preferences.** That's a nice fit for `PredictStrategy` — one prompt, one validated Python value. No custom Pydantic model needed here; the return type is just `list[str]`, and NOOA handles the parsing for us.

In [40]:
class TripScoutAgent(Agent):
    """Extract travel preferences from a request."""

    @strategy(PredictStrategy())
    async def extract_preferences(self, request: str) -> list[str]:
        """Return 4 to 6 concise preferences or constraints directly supported by the request."""
        ...


### Subagent 2

Next, we need am **itinerary builder**, a subagent that turns those preferences into an actual plan. We'll leave this method on the default strategy so it uses CodeAct: the model gets a Python REPL and can call helper methods on `self`.

The helpers are just ordinary Python and they give the CodeAct subagent a small surface for exploring the city guide.

In [41]:
class ItineraryBuilderAgent(Agent):
    """Build relaxed city itineraries from preferences and a small city guide."""

    def __init__(self, guide: list[dict], **kwargs):
        super().__init__(**kwargs)
        self._guide = guide

    def find_by_tag(self, tag: str, n: int = 5) -> list[dict]:
        """Return places whose tags contain tag, case-insensitive."""
        q = tag.lower()
        matches = [place for place in self._guide if any(q in t.lower() for t in place["tags"])]
        return matches[:n]

    def relaxed_options(self, max_minutes: int = 75) -> list[dict]:
        """Return places that fit a relaxed pace."""
        return [place for place in self._guide if place["duration"] <= max_minutes]

    def format_stop(self, place: dict, when: str) -> str:
        """Format one itinerary stop as a markdown bullet."""
        return f"- **{when}: {place['name']}** ({place['area']}) — {place['kind']}, about {place['duration']} min"

    async def build_itinerary(self, preferences: list[str], guide_label: str) -> str:
        """Build a relaxed one-day itinerary.

        Use self.find_by_tag(), self.relaxed_options(), and self.format_stop() to inspect the guide.
        Return markdown with a title, 3 to 5 timed stops, and a short note explaining the pacing.
        """
        ...


### The Parent Orchestrator

With both children in hand, let's compose them. `plan_day` has a real Python body, so we — not the LLM — decide the workflow order:

1. **First agent call.** Create the scout and `await` `extract_preferences`.
2. **Second agent call.** Create the itinerary builder and `await` `build_itinerary`, passing the preferences we just got back.
3. Store the handoff on `self.last_preferences` so we can inspect it later.

Nothing here is framework-specific — it's the same code you'd write to chain any two async functions. The only NOOA-specific bit is that the awaited methods happen to be agentic methods.

Notice that the child classes do **not** specify `llm=`. Because the parent creates them inside an active parent agent call, they inherit the parent's resolved model.

And notice too that this method instantiates the child classes directly — the LLM isn't deciding to spawn these subagents, we are.

In [42]:
class WeekendPlannerAgent(Agent, llm=model):
    """Coordinate subagents that turn a travel request into a one-day itinerary."""

    def __init__(self, guide: list[dict], **kwargs):
        super().__init__(**kwargs)
        self.guide = guide
        self.last_preferences: list[str] = []

    async def plan_day(self, request: str, city: str) -> str:
        """Run the subagent workflow and return a markdown itinerary."""
        scout = TripScoutAgent()
        preferences = await scout.extract_preferences(request)

        builder = ItineraryBuilderAgent(self.guide)
        itinerary = await builder.build_itinerary(preferences, guide_label=f"{city} city guide")

        self.last_preferences = preferences
        return itinerary


Time to run the workflow. From the outside, we only call the parent — it owns all the child-agent wiring.

In [43]:
planner = WeekendPlannerAgent(LISBON_GUIDE)

itinerary = await planner.plan_day(TRAVELER_REQUEST, city="Lisbon")

print(itinerary)


# Relaxed Saturday in Lisbon for Food, Books, Tiles & River Views

- **10:00: Manteigaria** (Chiado) — food, about 30 min
- **11:15: Museu Nacional do Azulejo** (Xabregas) — museum, about 90 min
- **13:15: Time Out Market** (Cais do Sodre) — food, about 60 min
- **15:15: Ler Devagar** (LX Factory) — bookshop, about 45 min
- **17:15: Miradouro de Santa Catarina** (Bica) — viewpoint, about 35 min

**Pacing note:** This keeps the day to five unhurried stops, with a pastry start, one deeper museum visit for tilework and history, an easy lunch, a bookshop-and-coffee pause, and a short sunset viewpoint. The route includes some gentle walking but avoids packing the whole day with hills, clubs, or late-night plans.

The intermediate handoff is still available for us to inspect.

In [44]:
print("Preferences passed from scout to itinerary builder:")
for preference in planner.last_preferences:
    print("-", preference)


Preferences passed from scout to itinerary builder:
- One Saturday in Lisbon with partner
- Interested in great food and bookshops
- Interested in tilework and a little history
- Wants river views
- Avoid clubs and late nights
- Prefers a relaxed schedule with some walking, but not all day

## 2. Running subagents in parallel

In real applications, it often happens that several subtasks are independent of each other and can run at the same time. Since subagents are just `Agent` instances and their methods are just `async` functions, we can parallelize them the way we would parallelize any async code in Python: with `asyncio.gather`. Again, there is no special construct — the parent method just awaits several coroutines at once.

Suppose our traveler is interested in several different aspects of the city — food, tilework, river views, bookshops — and we want a specialized scout to weigh in on each one. Rather than asking a single agent to cover everything, we spawn one small scout per interest category, run them concurrently, and let the parent gather the results and weave them into the itinerary.

One important rule to keep in mind: **use one subagent instance per concurrent task.** Agentic methods have an internal per-instance lock, so concurrent calls on the same instance serialize. Separate child instances can run independently.

In [45]:
import asyncio


class CategoryScoutAgent(Agent):
    """Suggest a few Lisbon picks for one interest category."""

    @strategy(PredictStrategy())
    async def suggest_picks(self, city: str, category: str) -> list[str]:
        """Return 2 or 3 picks in {city} for {category}. Each pick: 'Name — one-line reason'."""
        ...


class ParallelWeekendPlannerAgent(WeekendPlannerAgent):
    """Weekend planner with a parallel category-scouting step."""

    async def scout_categories(self, city: str, categories: list[str]) -> dict[str, list[str]]:
        """Ask one scout per category to suggest picks concurrently."""
        scouts = [CategoryScoutAgent() for _ in categories]
        results = await asyncio.gather(
            *(scout.suggest_picks(city, category) for scout, category in zip(scouts, categories))
        )
        return dict(zip(categories, results))


Each category gets its own `CategoryScoutAgent`, and all of their `suggest_picks(...)` calls are awaited together with `asyncio.gather`. This is standard Python asyncio — the only "agent" part is that the coroutines happen to be agentic methods. Because each scout is a separate instance, they run in parallel rather than serializing on a shared per-instance lock.

💡 Did you notice? We're just subclassing `WeekendPlannerAgent` below to add a new method — well, object-oriented agents. This has nothing to do with subagents; it's plain Python code reuse for the orchestrator itself. The subagent story is still the same: the parent instantiates children and awaits their methods.

In [46]:
parallel_planner = ParallelWeekendPlannerAgent(LISBON_GUIDE)

category_picks = await parallel_planner.scout_categories(
    city="Lisbon",
    categories=["great food", "tilework and history", "river views", "bookshops"],
)

for category, picks in category_picks.items():
    print(f"\n{category}:")
    for pick in picks:
        print(" -", pick)


great food:
 - Cervejaria Ramiro — A Lisbon classic for ultra-fresh seafood, especially garlic shrimp and clams.
 - Time Out Market Lisboa — A convenient hall featuring top local chefs and iconic Portuguese bites in one place.
 - Zé da Mouraria — Beloved for hearty, traditional Portuguese dishes with generous portions.

tilework and history:
 - Museu Nacional do Azulejo — The definitive Lisbon stop for centuries of Portuguese tilework, set in a historic convent.
 - Fronteira Palace — Lavish 17th-century palace with remarkable azulejo panels depicting battles, myths, and courtly life.
 - Igreja de São Roque — A richly historic church where ornate chapels and tile details reveal Lisbon’s religious and artistic heritage.

river views:
 - Miradouro de Santa Catarina — relaxed terrace views over the Tagus and the 25 de Abril Bridge.
 - MAAT Riverside — striking architecture with wide-open riverfront panoramas.
 - Cais das Colunas — classic waterfront spot for sunset views across the Tagus.

bookshops:
 - Livraria Bertrand — The world’s oldest operating bookshop, with atmospheric rooms and a central Chiado location.
 - Ler Devagar — A striking bookstore inside the LX Factory, known for its industrial setting, art, and browsing-friendly vibe.
 - Livraria Ferin — A historic family-run bookshop near Chiado with a strong literary heritage and curated selection.

This is still ordinary Python orchestration. The framework-specific part is just the child method: each `suggest_picks(...)` is an agentic method, and each child inherits the parent's model because it's created inside an active parent call.

## 3. Letting the LLM spawn subagents

So far, our parent methods were deterministic Python — we wrote out exactly which child agents to create. That's usually the clearest workflow design.

But there's another pattern that can turn useful: an agentic method can decide to create and call a subagent from *generated* code (similar to Claude Code's [dynamic workflows](https://code.claude.com/docs/en/workflows)). Since a CodeAct method just writes plain Python, and spawning a subagent is itself just plain Python (`child = ChildAgent(); await child.some_method(...)`), the model can do the same thing we've been doing by hand — as long as it knows the child classes exist.

That last part is the only wrinkle. The generated code runs against `self`, so each child class has to be reachable from `self` and has to show up in the model-facing surface (what you see with `doc(self)`). The idiomatic way to arrange that is a class attribute alias: `TripScoutAgent = TripScoutAgent`. It's not a dataclass field, just a name on the class that exposes the child so the generated code can reach it via `self.TripScoutAgent`.

To make the comparison with Part 1 direct, this adaptive planner will produce a full itinerary — same signature as `WeekendPlannerAgent.plan_day` — but this time the LLM chooses the order and composition of the two children.

In [50]:
class AdaptiveWeekendPlannerAgent(Agent, llm=model):
    """Planner whose CodeAct method decides at runtime how to compose its subagents."""

    TripScoutAgent = TripScoutAgent
    ItineraryBuilderAgent = ItineraryBuilderAgent

    def __init__(self, guide: list[dict], **kwargs):
        super().__init__(**kwargs)
        self.guide = guide

    async def plan_day(self, request: str, city: str) -> str:
        """Turn the traveler request into a one-day markdown itinerary for {city}.

        Two subagents are available:
        - `self.TripScoutAgent()` with `await scout.extract_preferences(request)`.
        - `self.ItineraryBuilderAgent(self.guide)` with `await builder.build_itinerary(preferences, guide_label)`.

        Decide the order and composition yourself, then return the final markdown itinerary.
        """
        ...


Let's run it. The call site looks identical to Part 1, but the internals are now LLM-driven: the parent method writes code, and that code chooses when to spawn each child.

In [ ]:
from nooa.tracing import set_session

set_session("adaptive_weekend_planner") # to better identify this trace in the trace viewer

adaptive_planner = AdaptiveWeekendPlannerAgent(LISBON_GUIDE)
itinerary = await adaptive_planner.plan_day(TRAVELER_REQUEST, city="Lisbon")
print(itinerary)


# Relaxed Saturday in Lisbon: tiles, books, food & river views

- **10:00: Manteigaria** (Chiado) — food, about 30 min
- **11:15: Museu Nacional do Azulejo** (Xabregas) — museum, about 90 min
- **13:30: Time Out Market** (Cais do Sodre) — food, about 60 min
- **15:00: Ler Devagar** (LX Factory) — bookshop, about 45 min
- **16:30: MAAT river walk** (Belem) — walk, about 60 min

**Pacing note:** This keeps the day to five gentle stops with breathing room between them for taxis/transit, coffee, and wandering without turning the day into a checklist. It starts with an easy classic pastry, gives the tile museum proper time, pauses for a flexible lunch, then ends with books and a calm riverside walk rather than late-night plans.

The output looks a lot like Part 1's itinerary — but the workflow that produced it was written by the model, not by us. Under the hood the CodeAct method wrote something like:

```python
scout = self.TripScoutAgent()
preferences = await scout.extract_preferences(request)

builder = self.ItineraryBuilderAgent(self.guide)
itinerary = await builder.build_itinerary(preferences, guide_label=f"{city} city guide")

return_result(itinerary)
```

Same two-step handoff as Part 1's `plan_day`, but the sequencing is a runtime decision. Open the trace viewer and inspect the parent's `execute_python` step to see the exact code the model generated — and the two child calls nested under it.

The tradeoff is what you'd expect. Static Python (Part 1) is predictable and debuggable; you always know what will run. LLM-driven composition (Part 3) is more flexible — the model can skip steps, reorder them, or add follow-up calls based on what it sees — but you give up some determinism. Reach for it when the right composition genuinely depends on the input.

Each subagent has its own surface too. If you instantiate a child directly in a notebook cell, pass `llm=model` explicitly — there's no active parent call to inherit from.

In [52]:
builder = ItineraryBuilderAgent(LISBON_GUIDE, llm=model)
print(doc(builder))


class ItineraryBuilderAgent:
    """Build relaxed city itineraries from preferences and a small city guide."""

    def find_by_tag(self, tag: str, n: int = 5) -> list[dict]:
        """Return places whose tags contain tag, case-insensitive."""
    def relaxed_options(self, max_minutes: int = 75) -> list[dict]:
        """Return places that fit a relaxed pace."""
    def format_stop(self, place: dict, when: str) -> str:
        """Format one itinerary stop as a markdown bullet."""
    async def build_itinerary(self, preferences: list[str], guide_label: str) -> str:
        """
        Build a relaxed one-day itinerary.
        
        Use self.find_by_tag(), self.relaxed_options(), and self.format_stop() to inspect the guide.
        Return markdown with a title, 3 to 5 timed stops, and a short note explaining the pacing.
        """

## Tracing the Handoff

If the trace viewer isn't already running, start it in a terminal:

```bash
nooa start-dev
```

Then re-run `await planner.plan_day(...)` and inspect the trace. You should see a parent call with two child calls underneath it: a Predict scout, then a CodeAct itinerary builder. The useful thing to inspect is the boundary between them — a messy request becomes a short `list[str]`, and only that list is handed to the builder.

## Recap

Three patterns, one framework primitive (a subagent is just another `Agent` instance):

- **Static handoff (Part 1):** the parent's Python fixes the order. Predictable, easy to debug, and usually the default choice.
- **Parallel fan-out (Part 2):** independent subtasks run concurrently with `asyncio.gather`. Remember the one-instance-per-concurrent-task rule from Part 2.
- **LLM-driven spawning (Part 3):** a CodeAct parent method decides at runtime whether to spawn a child. Expose the child class as a class attribute so it shows up in `doc(self)`.

And two rules that apply across all three:

- Children created inside an active parent call inherit the parent's resolved LLM. Outside a parent call, pass `llm=` explicitly.
- Subagents don't share memory, context blocks, or history. Treat each handoff like a function call: pass the data the next child needs.
